<a href="https://colab.research.google.com/github/aldo02032004/NER_env/blob/main/notebooks/01_label_entities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Label entities: GLiNER + local LLM validation → training data

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cell 1. If Colab asks to restart, restart and continue from cell 2.
3. Fill in the form in cell 2, select it, then **Runtime → Run cell and below**.

The LLM (Gemma 2 9B SahabatAI, GGUF, ~5.8 GB) is downloaded once per session in cell 6 and runs on the GPU, so no API key is needed.

Runtime disconnected? Run everything again: GLiNER predictions are cached and finished documents are not sent again. `llm_budget_per_run` reached? Re-run cell 6 only.

In [ ]:
# @title 1. Install (once per session)
!git -C /content/NER_env pull -q 2>/dev/null || git clone -q https://github.com/aldo02032004/NER_env.git /content/NER_env
%pip install -q -r /content/NER_env/requirements.txt
# kalau paket yang sudah ter-import ikut di-update, Colab sendiri menampilkan tombol "Restart session"

In [ ]:
# @title 2. Form and config
# @markdown **PROJECT**: short name (lowercase, no spaces), used for file names and the `project` field in the training data.
PROJECT = "prabowo_vladivostok"  # @param {type:"string"}
# @markdown **LINKS**: Google Sheets / Drive links, comma-separated, shared as *Anyone with the link*.
LINKS = ""  # @param {type:"string"}
# @markdown **CONFIG_PATH**: config shared by all projects (labels, thresholds, aliases).
CONFIG_PATH = "/content/drive/MyDrive/ner/config.json"  # @param {type:"string"}

import re, sys
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/NER_env")
from ner.config import load_config

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
if not PROJECT or not LINKS.strip():
    raise ValueError("[ERROR] Isi PROJECT dan LINKS di form dulu")

cfg = load_config(CONFIG_PATH)
THRESHOLD = cfg["ner_review_threshold"][cfg["active_model"]]
print(f"[OK] PROJECT = {PROJECT} | label: {cfg['labels']}")
print(f"[INFO] GLiNER {cfg['active_model']} (threshold review {THRESHOLD}) | LLM {cfg['llm_model_file']} "
      f"(batch {cfg['batch_size']}, batas per run {cfg['llm_budget_per_run']})")

In [ ]:
# @title 3. Load spreadsheet and clean text (no rows removed)
from ner.io import load_sheet, prepare_frame

df = prepare_frame(load_sheet(LINKS))
flags = ["is_media", "is_retweet", "too_short", "is_duplicate"]
print(f"[INFO] {len(df)} baris | ditandai: { {f: int(df[f].sum()) for f in flags} }")
ready = df[df["siap_ner"]]
docs = dict(zip(ready["doc_id"], ready["text"]))
print(f"[OK] {len(docs)} dari {len(df)} baris siap NER")
display(df[["text_raw", "text", *flags, "siap_ner"]].head(10))

In [ ]:
# @title 4. GLiNER predictions (cached, resumable)
import torch
from gliner import GLiNER
from ner.formats import spans_to_names
from ner.predict import predict_all, preds_cache_path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("[WARN] GLiNER jalan di CPU, bisa BERJAM-JAM. Ganti ke T4 GPU lalu mulai lagi dari cell 1.")
gliner_model = GLiNER.from_pretrained(cfg["active_model"]).to(DEVICE)

preds = predict_all(gliner_model, docs, cfg, preds_cache_path(cfg, PROJECT))
print(f"[OK] {sum(map(len, preds.values()))} entitas, {sum(not s for s in preds.values())} doc tanpa entitas")
for d in list(docs)[:5]:
    ents = {k: v for k, v in spans_to_names(docs[d], preds[d], cfg["labels"]).items() if v}
    print(f"\n- {docs[d][:150]}\n  {ents}")

In [ ]:
# @title 5. Routing: which documents the LLM checks
from ner.routing import REASONS, doc_score, route_all

routes = route_all(docs, preds, cfg)
for reason in REASONS:
    for d in [d for d, r in routes.items() if r == reason][:2]:
        print(f"[{reason}] skor={doc_score(preds[d])} | {docs[d][:120]}")

In [ ]:
# @title 6. LLM validation → training data (resumable)
from ner.llm import log_path, make_generate, print_summary, review_all, summarize_log
from ner.schema import read_jsonl

if "generate" not in globals():   # model dimuat sekali; kalau dimuat ulang, memori GPU habis
    print(f"[INFO] Memuat {cfg['llm_model_file']} (unduhan pertama ~5.8 GB)...")
    generate = make_generate(cfg)
LOG, MASTER = log_path(cfg, PROJECT), cfg["paths"]["master_data"]

finished = review_all(docs, preds, routes, cfg, PROJECT, generate, MASTER, LOG)
log_rows = read_jsonl(LOG)
print_summary(summarize_log(log_rows, routes))
print(f"[INFO] Data latih proyek ini di master: {sum(r['project'] == PROJECT for r in read_jsonl(MASTER))} doc")
for r in [r for r in log_rows if r["status"] == "missing" and r["doc_id"] in routes][:5]:
    print(f"[missing] {r['missing']} | {docs[r['doc_id']][:120]}")
print("[OK] Selesai" if finished else "[STOP] Belum selesai, jalankan ulang cell ini untuk lanjut")

In [ ]:
# @title 7. Export per project
import os
from ner.io import build_export
from ner.llm import llm_entities

# entitas GLiNER yang belum dicek LLM hanya dipakai kalau skornya >= threshold
out = build_export(df, docs, preds, llm_entities(read_jsonl(LOG)), cfg["labels"],
                   cfg["aliases"].get(PROJECT, {}), min_score=THRESHOLD)
out_dir = cfg["paths"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)
out.to_pickle(f"{out_dir}/{PROJECT}_entities.pkl")   # kolom list tetap list
out.to_csv(f"{out_dir}/{PROJECT}_entities.csv", index=False)
print(f"[OK] {len(out)} baris ditulis ke {out_dir}/{PROJECT}_entities.pkl dan .csv")
print(f"[INFO] ner_source: {out['ner_source'].value_counts(dropna=False).to_dict()}")
for lab in cfg["labels"]:
    names = out[f"entities_{lab}"].explode().dropna()
    print(f"  {lab:<20} {len(names):>6} sebutan | teratas: {names.value_counts().head(5).to_dict()}")
display(out.head(10))